# 07. Jenkins connector real pipeline run

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path
from urllib.parse import urljoin

import httpx
from deepagents import create_deep_agent
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "pyproject.toml").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

DEFAULT_MODEL = "openrouter:openai/gpt-oss-120b:free"
PIPELINE_PARAMETERS = {"OPENCLAW_SMOKE": "true"}
RUN_REAL_PIPELINE = os.getenv("OPENCLAW_RUN_REAL_JENKINS_PIPELINE") == "1"


def model_name() -> str:
    return os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL)


def pretty(payload: str | dict) -> None:
    if isinstance(payload, str):
        try:
            payload = json.loads(payload)
        except json.JSONDecodeError:
            print(payload)
            return
    print(json.dumps(payload, ensure_ascii=False, indent=2))


## 1. Проверить, что Jenkins tools подключены

Эта ячейка не ходит в сеть. Она проверяет, что агент получит `get_jenkins_job_info` и `trigger_jenkins_job`.


In [ ]:
from connectors import CONNECTOR_TOOLS

tool_names = [tool.name for tool in CONNECTOR_TOOLS]
jenkins_tool_names = [name for name in tool_names if "jenkins" in name]

print(jenkins_tool_names)
assert {"get_jenkins_job_info", "trigger_jenkins_job"} <= set(tool_names)


## 2. Собрать агента с Jenkins tools

Это smoke-check того, что Deep Agents graph собирается с тем же набором tools, который используется в основном проекте.


In [ ]:
BASE_PROMPT = """\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, implementation, and DevOps checks.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- for Jenkins, this is a test environment, so real job reads and smoke builds are allowed when requested.
"""

agent = create_deep_agent(
    model=model_name(),
    tools=CONNECTOR_TOOLS,
    system_prompt=BASE_PROMPT,
)

print(type(agent).__name__)


## 3. Preview pipeline request

Перед реальным запуском смотрим точный Jenkins request. Эта ячейка не запускает job.


In [ ]:
from connectors.jenkins import trigger_jenkins_job

dry_run_result = trigger_jenkins_job.invoke(
    {
        "parameters": PIPELINE_PARAMETERS,
        "dry_run": True,
    }
)

pretty(dry_run_result)
assert "\"dry_run\": true" in dry_run_result
assert "buildWithParameters" in dry_run_result


## 4. Read Jenkins job info

Эта ячейка ходит в Jenkins API. Если endpoint закрыт или нет сети, notebook должен упасть здесь, а не в момент реального trigger.


In [ ]:
from connectors.jenkins import get_jenkins_job_info

job_info_result = get_jenkins_job_info.invoke({})
pretty(job_info_result)

job_info = json.loads(job_info_result)
assert job_info.get("ok") is True, job_info_result


## 5. Реально запустить pipeline/job

Эта ячейка выполняет мутацию в Jenkins только когда `OPENCLAW_RUN_REAL_JENKINS_PIPELINE=1`.


In [ ]:
if not RUN_REAL_PIPELINE:
    raise RuntimeError(
        "Real Jenkins pipeline run is disabled. Set OPENCLAW_RUN_REAL_JENKINS_PIPELINE=1 in .env when you intentionally want to trigger it."
    )

real_build_result = trigger_jenkins_job.invoke(
    {
        "parameters": PIPELINE_PARAMETERS,
        "dry_run": False,
    }
)

pretty(real_build_result)
trigger_payload = json.loads(real_build_result)
assert trigger_payload.get("ok") is True, real_build_result


## 6. Дождаться Jenkins queue и build result

После trigger Jenkins обычно возвращает queue URL. Эта ячейка ждет, пока queue item превратится в build, затем опрашивает build API до результата.


In [ ]:
from connectors.jenkins import _auth

queue_url = trigger_payload.get("queue_url")
if not queue_url:
    raise RuntimeError(f"Jenkins did not return queue_url: {trigger_payload}")

queue_api_url = urljoin(queue_url.rstrip("/") + "/", "api/json")

with httpx.Client(auth=_auth(), timeout=20.0) as client:
    executable = None
    for attempt in range(30):
        queue_response = client.get(queue_api_url)
        queue_response.raise_for_status()
        queue_payload = queue_response.json()
        executable = queue_payload.get("executable")
        if executable:
            break
        print(f"queue wait {attempt + 1}/30: {queue_payload.get(why) or waiting}")
        time.sleep(2)

    if not executable:
        raise TimeoutError(f"Jenkins queue did not produce executable: {queue_payload}")

    build_url = executable["url"]
    build_api_url = urljoin(build_url.rstrip("/") + "/", "api/json")
    print(f"Build URL: {build_url}")

    for attempt in range(90):
        build_response = client.get(build_api_url)
        build_response.raise_for_status()
        build_payload = build_response.json()
        if not build_payload.get("building", False):
            break
        print(f"build wait {attempt + 1}/90: still running")
        time.sleep(5)

pretty({
    "build_url": build_url,
    "building": build_payload.get("building"),
    "result": build_payload.get("result"),
    "duration": build_payload.get("duration"),
})

assert build_payload.get("building") is False, build_payload
assert build_payload.get("result") in {"SUCCESS", "UNSTABLE"}, build_payload


## Prompt для UI

Можно повторить тот же сценарий через Deep Agents UI:

> Use the Jenkins connector. First list which Jenkins tools are available. Then read the configured Jenkins job info. After that trigger a real smoke build for the configured job with parameter OPENCLAW_SMOKE=true and summarize the queue/build response.

Для безопасной демонстрации замени `trigger a real smoke build` на `preview the request in dry-run mode`.
